In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import torchvision
import timm
import torchxrayvision as xrv
import copy
import os

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


## Loading the Data

In [3]:
# Data Transforms
input_size  = 224
batch_sizes = {'vit_base_patch16_224' : 32, 'vit_tiny_patch16_224' : 128, 'resnet50' : 64}
models      = ('vit_base_patch16_224', 'vit_tiny_patch16_224', 'resnet50')

mean = -5.6936
std = 534.7731

transforms = torchvision.transforms.Compose([
    xrv.datasets.XRayCenterCrop(), 
    xrv.datasets.XRayResizer(input_size), 
    xrv.datasets.transforms.ToTensor(), 
    xrv.datasets.transforms.Normalize(mean, std)])

model_save_path = 'E:/Models/'
path            = 'E:/Datasets/rsna-pneumonia-detection-challenge/stage_2_train_images'

dataset = xrv.datasets.RSNA_Pneumonia_Dataset(imgpath=path, extension='.dcm', transform=transforms)

splits = (11511, 1500, 1500)

train_dataset, val_dataset, test_dataset = random_split(dataset, splits, torch.Generator().manual_seed(42))

## Loading the Model

In [9]:
def train(model, model_name, dataloaders, criterion, optimizer, scheduler=None, num_epochs=25, device='cuda'):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    
    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        for phase in ('train', 'val'):
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.
            running_corrects = 0
            num_inputs = 0.
            for data in dataloaders[phase]:
                inputs = data['img'].permute(0, 2, 3, 1).to(device)
                labels = data['lab'][:, 0].to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels.long())
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                num_inputs += inputs.size(0)
            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / num_inputs
            epoch_acc  = running_corrects.double() / num_inputs

            print(f'{phase} Loss: {epoch_loss} | Acc: {epoch_acc}')

            if phase == 'val' and epoch_loss < best_loss:
                best_loss = epoch_loss
                best_model_wts = copy.deepcopy(model.state_dict())
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
        print()

    print(f'Model: {model_name} | Best val Acc: {best_acc:4f} | loss: {best_loss:4f}')

    # load best model weights
    model.load_state_dict(best_model_wts)
    return model

In [10]:
def test(model, dataloaders, device='cuda'):
    with torch.no_grad():
        n_correct = 0
        n_samples = 0

        for data in dataloaders['test']:

            features = data['img'].permute(0, 2, 3, 1).to(device)
            labels   = data['lab'][:, 0].to(device)

            outputs = model(features)
            _, predictions = torch.max(outputs, 1)
            n_samples += labels.shape[0]
            n_correct += (predictions == labels).sum().item()

        return 100 * n_correct / n_samples

In [ ]:
for model_name in models:

    batch_size   = batch_sizes[model_name]
    train_loader = DataLoader(train_dataset, batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size, shuffle=True)
    test_loader  = DataLoader(test_dataset, batch_size, shuffle=False)

    dataloaders  = {'train':train_loader, 'val':val_loader, 'test':test_loader}

    model = timm.create_model(model_name, pretrained=True, in_chans=1, num_classes=2).to(device)

    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0.9)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

    model = train(model, model_name, dataloaders, criterion, optimizer, scheduler)

    torch.save(model, f'{model_save_path}{model_name}.pth')

In [13]:
# test(model, dataloaders)

89.0